# LaringoPDI — executor de shards no Colab

**Antes de começar:** Ambiente de execução → Alterar tipo → **GPU T4**.

1. Edite **só** a célula abaixo (`SHARD_ID` e, se o Nathan pedir, `STAGE`).
2. Menu *Ambiente de execução → Executar tudo*.
3. Pode fechar a aba quando a última célula mostrar `FIM`. Se a sessão cair, rode de novo: jobs concluídos são pulados.

Pasta compartilhada do Drive: `MyDrive/LaringoPDI/` com `code/` (repositório), `cache/` (imagens pré-processadas) e `runs/` (resultados).
Os dados são do BAGLS (CC BY-NC-SA 4.0): **não copie para fora da pasta do projeto.**

In [ ]:
SHARD_ID = 0          # 0 Nathan · 1 Marcos Vinicius · 2 Pedro · 3 Marcos Mendes · 4 Tales
STAGE = "stage1"      # stage1 (SimpleCNN rápida) ou stage2 (fine-tuning)
MEASURE_ONLY = False  # True = só mede 1 job (orçamento da Fase 4) e para

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/LaringoPDI'
import os, subprocess
assert os.path.isdir(f'{DRIVE}/code'), 'Pasta LaringoPDI/code não encontrada no Drive (peça ao Nathan o atalho).'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# código + dependências (torch, sklearn, skimage e opencv já vêm no Colab)
!rm -rf /content/LaringoPDI && cp -r "$DRIVE/code" /content/LaringoPDI
%cd /content/LaringoPDI
!pip -q install -e . --no-deps
!pip -q install pyyaml imagehash grad-cam > /dev/null

In [ ]:
# caches necessários a este shard: copia do Drive para o disco local (mais rápido) e liga runs/ ao Drive
import json, glob, shutil
os.makedirs('data/processed/cache', exist_ok=True)
os.makedirs(f'{DRIVE}/runs', exist_ok=True); os.makedirs(f'{DRIVE}/logs', exist_ok=True)
if not os.path.islink('results/runs'):
    shutil.rmtree('results/runs', ignore_errors=True); os.symlink(f'{DRIVE}/runs', 'results/runs')
shards = json.load(open(f'results/shards/{STAGE}.json'))
need = ['E00'] if MEASURE_ONLY else shards['caches'][str(SHARD_ID)]
shutil.copy(f'{DRIVE}/cache/frame_labels.csv', 'data/processed/cache/')
for e in need:
    for f in glob.glob(f'{DRIVE}/cache/{e}_*'):
        if not f.endswith('.probe.npy'):
            shutil.copy(f, 'data/processed/cache/')
print('shard', SHARD_ID, '| experimentos:', need, '| jobs:', len(shards['shards'][str(SHARD_ID)]))

In [ ]:
# medição do orçamento (Fase 4): 1 job de cada modelo em E00, tarefa A, fold 0
if MEASURE_ONLY:
    for m in ('simplecnn_fast', 'resnet18'):
        !python -m laringopdi train --exp E00 --task A --model {m} --fold 0 --seed 0 > "{DRIVE}/logs/measure_{m}.log" 2>&1
        r = json.load(open(f'results/runs/E00/A/{m}/fold0_seed0/metrics.json'))
        print(m, '| épocas:', r['epochs_run'], '| s/época:', round(sum(h['sec'] for h in r['history']) / r['epochs_run'], 1),
              '| total (s):', round(r['seconds']), '| device:', r['device'])

In [ ]:
if not MEASURE_ONLY:
    log = f'{DRIVE}/logs/{STAGE}_shard{SHARD_ID}.log'
    !python -m laringopdi train --stage {STAGE} --shard {SHARD_ID} > "{log}" 2>&1
    !tail -n 3 "{log}"
print('FIM')